### **SavedModel 格式的用途详解**

#### **1. SavedModel 是什么？**  
SavedModel 是 TensorFlow 推荐的模型保存格式，用于**完整保存训练好的模型**，包含以下内容：  
- **模型架构**：网络层结构、连接方式等。  
- **模型权重**：训练得到的参数值。  
- **计算图**：前向传播和反向传播的运算逻辑。  
- **训练配置**：优化器（如 Adam）、损失函数、评估指标等。  
- **签名（Signatures）**：定义模型的输入输出接口（如推理函数 `predict`）。  

#### **2. 为何要使用 SavedModel 格式？**  
##### **(1) 生产环境部署**  
- **TensorFlow Serving**：专为生产环境设计的服务框架，直接加载 SavedModel 提供高性能推理服务（REST/gRPC API）。  
- **跨平台兼容性**：支持在多种环境中使用，如云端服务器、边缘设备等。  

##### **(2) 功能完整性**  
- **支持自定义层和复杂模型**：相比 HDF5（`.h5`），SavedModel 能更好地保存包含自定义层或子类化模型的复杂结构。  
- **版本控制**：可保存多个模型版本（如 `exported_model/1/`, `exported_model/2/`），方便回滚和更新。  

##### **(3) 工具链集成**  
- **TensorFlow Lite**：可将 SavedModel 转换为轻量级格式（`.tflite`），部署到移动端或 IoT 设备。  
- **TensorFlow.js**：转换为 JavaScript 格式，支持浏览器端推理。  

---


#### **3. SavedModel 与 HDF5 格式的区别**  
| **特性**                | **SavedModel**                  | **HDF5 (`.h5`)**                |
|-------------------------|---------------------------------|---------------------------------|
| **保存内容**            | 完整模型（含计算图和签名）      | 模型架构+权重（部分训练配置）   |
| **自定义模型支持**      | 支持复杂模型（如子类化模型）    | 有限支持（可能丢失部分信息）    |
| **生产部署**            | 直接用于 TensorFlow Serving     | 需转换格式                      |
| **文件结构**            | 目录（多文件）                  | 单个文件（`.h5`）               |
| **TensorFlow 版本兼容性** | 专为 TF 2.x 优化               | 兼容早期版本（如 TF 1.x）       |

---

#### **4. 使用场景示例**  
##### **(1) 部署到 TensorFlow Serving**  
1. 保存模型：  
   ```python
   model.save('exported_model/', save_format='tf')
   ```  
2. 启动 TensorFlow Serving 服务：  
   ```bash
   tensorflow_model_server --rest_api_port=8501 \
   --model_name=leaf_disease \
   --model_base_path=/path/to/exported_model/
   ```  
3. 通过 REST API 调用：  
   ```python
   import requests
   data = {"instances": [input_data.tolist()]}  # 输入数据需预处理为模型接受的格式
   response = requests.post("http://localhost:8501/v1/models/leaf_disease:predict", json=data)
   predictions = response.json()["predictions"]
   ```

##### **(2) 转换为 TensorFlow Lite**  
```python
converter = tf.lite.TFLiteConverter.from_saved_model('exported_model/')
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
```

---

#### **5. 注意事项**  
- **避免手动修改文件**：SavedModel 内部文件依赖严格的结构，手动修改可能导致加载失败。  
- **版本管理**：生产环境中建议使用目录版本号（如 `exported_model/1/`），便于无缝更新模型。  
- **依赖项一致**：在不同环境中加载 SavedModel 时，需确保 TensorFlow 版本和依赖库一致。  

---

### **总结**  
通过 `model.save('exported_model/', save_format='tf')` 保存的 **SavedModel 格式** 是 TensorFlow 生态中的“黄金标准”，尤其适合：  
- 生产环境部署（如 TensorFlow Serving）。  
- 跨平台转换（移动端、浏览器端）。  
- 复杂模型（含自定义逻辑）的完整保存。  

如果你的目标是**将模型投入实际应用**，优先选择 SavedModel；
若仅需本地快速保存/加载简单模型，HDF5 格式（`.h5`）仍可满足需求。